# Exploratory analyses

Not part of the pre-planned analysis of Section 3: here we look for patterns behind the results, to learn *when*
and *for which items* typed decision models are swayed. Everything is computed from the stored answers (no API calls).
Each section says what it asks. Fallacy oversight is left out throughout, since there a changed verdict is correct.
Observations are collected in the Notes at the end.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import subprocess
import sys
from itertools import combinations
from pathlib import Path
CODE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CODE))   # judgebias and tdm
import numpy as np
import pandas as pd
from judgebias import explore, metrics, run
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)

d = explore.long_table()
explore.save(d)                       # for the R scripts
d = d[d.bias != "fallacy"]
calm = d[d.format == "calm"].dropna(subset=["flip"]).copy()
print(f"{len(d):,} perturbed prompts (8 models x 3 formats), {len(calm):,} with a verdict in the CALM format")

306,024 perturbed prompts (8 models x 3 formats), 100,668 with a verdict in the CALM format


## 1. When do verdicts flip? How decided the judge was on the original

`margin` = |P(R1) - 0.5| on the original prompt: 0 = undecided, 0.5 = certain. Share of flipped verdicts per margin
bin, CALM format.

In [ ]:
calm["margin_bin"] = pd.cut(calm.margin, [0, .1, .2, .3, .4, .49, .5], include_lowest=True)
print("per model"); display(calm.pivot_table(index="margin_bin", columns="model", values="flip", aggfunc="mean", observed=True).round(2))
print("per test"); calm.pivot_table(index="margin_bin", columns="bias", values="flip", aggfunc="mean", observed=True).round(2)

per model


model,clef,clef-flash,d1,jev-1.13,kev-4b,pplx-decider-v1.1-27b,solar-decide,tev1-4b
margin_bin,,,,,,,,
"(-0.001, 0.1]",0.43,0.37,0.32,0.36,0.30,0.28,0.45,0.26
"(0.1, 0.2]",0.33,0.26,0.10,0.19,0.12,0.16,0.35,0.13
"(0.2, 0.3]",0.21,0.15,0.07,0.13,0.08,0.12,0.34,0.05
"(0.3, 0.4]",0.19,0.08,0.07,0.06,0.06,0.06,0.12,0.03
"(0.4, 0.49]",0.08,0.03,0.06,0.06,0.03,0.06,0.09,0.04
"(0.49, 0.5]",0.04,NaN,0.02,0.07,0.00,0.00,0.15,0.00


per test


bias,authority,"authority, better response",bandwagon,compassion,distraction,diversity,"diversity, neutral note",position,sentiment,verbosity,"verbosity, better response"
margin_bin,,,,,,,,,,,
"(-0.001, 0.1]",0.40,0.42,0.41,0.37,0.24,0.26,0.23,0.41,0.49,0.20,0.54
"(0.1, 0.2]",0.20,0.42,0.30,0.17,0.06,0.07,0.07,0.28,0.44,0.11,0.30
"(0.2, 0.3]",0.14,0.31,0.20,0.10,0.03,0.03,0.03,0.18,0.42,0.04,0.14
"(0.3, 0.4]",0.10,0.16,0.11,0.04,0.02,0.02,0.02,0.15,0.38,0.05,0.11
"(0.4, 0.49]",0.05,0.14,0.07,0.02,0.01,0.01,0.01,0.06,0.26,0.01,0.03
"(0.49, 0.5]",0.11,0.12,0.06,0.08,0.04,0.04,0.06,0.08,0.29,0.04,0.02


## 2. Are the same items vulnerable for all models?

For each test, the mean shift (and flip rate) of every item per model, then the Spearman correlation between every
pair of the 8 models, averaged. High = the same items move all models (a property of the item or its perturbation);
low = each model has its own weak items.

In [ ]:
out = []
for bias, g in d[d.format == "calm"].groupby("bias"):
    row = {"test": bias}
    for col in ["shift", "flip"]:
        w = g.groupby(["item", "model"])[col].mean().unstack()
        row[col] = np.nanmean([w[a].corr(w[b], method="spearman") for a, b in combinations(w.columns, 2)])
    out.append(row)
pd.DataFrame(out).set_index("test").round(2)

.venv/lib/python3.13/site-packages/pandas/core/nanops.py:1673: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  return spearmanr(a, b)[0]


,shift,flip
test,,
authority,0.41,0.21
"authority, better response",0.35,0.21
bandwagon,0.29,0.13
compassion,0.14,0.05
distraction,0.11,0.04
diversity,0.28,0.13
"diversity, neutral note",0.27,0.09
position,0.17,0.18
sentiment,0.57,0.51


## 3. Question source

Flip rate per source dataset of the item (CALM format, all models). Alignment items come from five preference
datasets, fact items from three question sets (Section 3).

In [ ]:
calm.pivot_table(index="source", columns="bias", values="flip", aggfunc="mean").round(2)

bias,authority,"authority, better response",bandwagon,compassion,distraction,diversity,"diversity, neutral note",position,sentiment,verbosity,"verbosity, better response"
source,,,,,,,,,,,
Emerton-DPO,0.18,0.27,0.24,0.13,0.07,0.07,0.06,0.25,NaN,NaN,NaN
GSM8K,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.33,0.03,0.10
MATH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.31,0.02,0.04
Orca-DPO,0.12,0.24,0.18,0.11,0.06,0.06,0.05,0.17,NaN,NaN,NaN
Py-DPO,NaN,NaN,0.10,0.08,0.04,0.04,0.04,0.12,NaN,NaN,NaN
Roleplay,NaN,NaN,0.14,0.06,0.02,0.03,0.05,0.08,NaN,NaN,NaN
ScienceQA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.29,0.03,0.06
Truthy-DPO,NaN,NaN,0.15,0.11,0.05,0.07,0.06,0.16,NaN,NaN,NaN


## 4. Was the judge right or wrong on the original?

Flip rate when the judge picked the better (`1`) or the worse (`0`) response on the original. A perturbation that
pushes *away* from the response it targets flips verdicts that were on that response.

In [ ]:
calm.pivot_table(index="orig_correct", columns="bias", values="flip", aggfunc="mean").round(2)

bias,authority,"authority, better response",bandwagon,compassion,distraction,diversity,"diversity, neutral note",position,sentiment,verbosity,"verbosity, better response"
orig_correct,,,,,,,,,,,
0.0,0.43,0.02,0.02,0.07,0.09,0.10,0.08,0.28,0.42,0.42,0.47
1.0,0.02,0.38,0.23,0.11,0.04,0.04,0.04,0.12,0.31,0.02,0.06


## 5. Length of the two responses

Flip rate by the length of the better response relative to the worse one (quartiles of log(len R1 / len R2)).

In [ ]:
calm["length"] = pd.qcut(calm.log_len_ratio, 4, labels=["R1 much shorter", "R1 shorter", "R1 longer", "R1 much longer"])
calm.pivot_table(index="length", columns="bias", values="flip", aggfunc="mean", observed=True).round(2)

bias,authority,"authority, better response",bandwagon,compassion,distraction,diversity,"diversity, neutral note",position,sentiment,verbosity,"verbosity, better response"
length,,,,,,,,,,,
R1 much shorter,0.18,0.22,0.14,0.10,0.06,0.06,0.06,0.17,0.28,0.02,0.04
R1 shorter,0.20,0.22,0.18,0.12,0.06,0.07,0.07,0.22,0.33,0.03,0.08
R1 longer,0.13,0.34,0.19,0.12,0.05,0.05,0.06,0.18,0.28,0.02,0.06
R1 much longer,0.13,0.27,0.15,0.08,0.04,0.04,0.03,0.11,0.34,0.04,0.07


## 6. Bandwagon: does a larger claimed majority pull harder?

Mean shift towards the worse response per claimed majority (CALM format), and the slope per 10 points of majority.

In [ ]:
bw = d[(d.format == "calm") & (d.bias == "bandwagon")].dropna(subset=["shift"]).copy()
bw["share"] = bw.variant.astype(int)
display(bw.pivot_table(index="share", columns="model", values="shift", aggfunc="mean").round(3))
pd.Series({m: 10 * np.polyfit(g.share, g["shift"], 1)[0] for m, g in bw.groupby("model")}, name="slope per 10 points").round(3)

model,clef,clef-flash,d1,jev-1.13,kev-4b,pplx-decider-v1.1-27b,solar-decide,tev1-4b
share,,,,,,,,
60,0.188,0.069,0.032,0.141,0.051,0.126,0.033,0.008
70,0.256,0.109,0.045,0.174,0.070,0.177,0.045,0.025
80,0.301,0.119,0.050,0.186,0.092,0.206,0.054,0.046
90,0.302,0.138,0.053,0.199,0.126,0.248,0.090,0.074


clef                     0.039
clef-flash               0.022
d1                       0.007
jev-1.13                 0.019
kev-4b                   0.025
pplx-decider-v1.1-27b    0.040
solar-decide             0.018
tev1-4b                  0.022
Name: slope per 10 points, dtype: float64

## 7. Model level: are more accurate or more consistent models less biased?

One row per model (CALM format, the 10 bias tests): mean excess, mean size of the shift, consistency, accuracy on
the originals. Only 8 models, so this is descriptive.

In [ ]:
res = pd.read_csv(CODE / "tables" / "main.csv")
res = res[res.format == "calm"]                   # all eleven tests; fallacy oversight on the bias scale
m = res.groupby("model").agg(mean_excess=("excess", "mean"), mean_abs_shift=("shift", lambda s: s.abs().mean()),
                             CR=("CR", "mean"), accuracy=("acc_before", "mean"))
display(m.round(3))
practice = pd.read_csv(CODE / "tables" / "practice.csv")     # accuracy and calibration on the original alignment prompts
NAMES = {"Jev 1.13": "jev-1.13", "Kev 4B": "kev-4b", "Tev1 4B": "tev1-4b", "Solar Decide": "solar-decide", "D1": "d1",
         "Perplexity Decider": "pplx-decider-v1.1-27b", "Clef": "clef", "Clef Flash": "clef-flash"}
m = m.join(practice.assign(model=practice.judge.map(NAMES)).dropna(subset=["model"]).set_index("model")[["acc_al", "ece", "brier"]])
print("Spearman with the mean excess: accuracy (alignment) %.2f, CR %.2f, ECE %.2f, Brier %.2f" % tuple(
    m[c].corr(m.mean_excess, method="spearman") for c in ["acc_al", "CR", "ece", "brier"]))

,mean_excess,mean_abs_shift,CR,accuracy
model,,,,
clef,0.118,0.181,1.000,0.825
clef-flash,0.106,0.130,1.000,0.731
d1,0.117,0.153,0.998,0.825
jev-1.13,0.109,0.180,0.990,0.841
kev-4b,0.134,0.135,1.000,0.758
pplx-decider-v1.1-27b,0.112,0.157,1.000,0.823
solar-decide,0.078,0.135,0.932,0.756
tev1-4b,0.117,0.129,1.000,0.761


Spearman with the mean excess: accuracy (alignment) -0.07, CR 0.52, ECE -0.36, Brier -0.24


## 8. Mixed models (R, lme4)

`R/flips.R`: a logistic mixed model of whether a verdict flips, with a random effect per item, so that the many
prompts of one item are not treated as independent. Odds ratios: above 1 = more flips. `margin10` is per 0.1 of
margin. Reference levels: Jev, the CALM format, the neutral note, and the first source alphabetically.

`R/shifts.R`: a linear mixed model with crossed random effects, used only to split the variance of the shift into
parts due to the test, the model, the format, the item and their combinations.

In [ ]:
for script in ["R/flips.R", "R/shifts.R"]:
    out = subprocess.run(["Rscript", script], cwd=CODE, capture_output=True, text=True)
    print(out.stdout, out.stderr[-2000:] if out.returncode else "")

== 1. CALM format: flip ~ margin + original correct + length ratio + bias + model + source + (1 | item)
                               odds_ratio     lo     hi
(Intercept)                         0.280  0.224  0.349
margin10                            0.465  0.454  0.476
orig_correct                        1.540  1.441  1.647
log_len_ratio                       0.954  0.879  1.037
biasauthority                       3.073  2.636  3.584
biasauthority, better response      6.717  5.807  7.768
biasbandwagon                       4.674  4.128  5.292
biascompassion                      2.325  1.970  2.743
biasdistraction                     0.984  0.839  1.155
biasdiversity                       1.084  0.960  1.224
biasposition                        4.652  3.995  5.418
biassentiment                      18.296 13.808 24.243
biasverbosity                       0.937  0.678  1.293
biasverbosity, better response      2.539  1.903  3.388
modelclef                           2.170  1.972  2.387



== signed shift, all formats (n = 306023, 374 s)
          grp     vcov share
     Residual 2.09e-02 0.567
         bias 9.30e-03 0.252
    item:bias 2.95e-03 0.080
   bias:model 2.23e-03 0.060
   item:model 7.48e-04 0.020
        model 5.14e-04 0.014
  bias:format 1.43e-04 0.004
 model:format 4.84e-05 0.001
       source 4.16e-05 0.001
         item 0.00e+00 0.000
       format 4.68e-06 0.000

== size of the shift |p_hat - p|, all formats (n = 306023, 654 s)
          grp     vcov share
     Residual 1.51e-02 0.574
         bias 3.64e-03 0.138
   item:model 2.38e-03 0.090
   bias:model 1.92e-03 0.073
    item:bias 1.40e-03 0.053
         item 1.31e-03 0.050
 model:format 1.49e-04 0.006
       source 1.58e-04 0.006
        model 1.33e-04 0.005
  bias:format 1.02e-04 0.004
       format 2.14e-05 0.001
 


## 9. A minimum size for a bias

With nearly deterministic judges, even an excess of 0.01 is significant (Section 3). To call something a bias that
matters, we need a smallest effect of interest δ. Anchor: how often a verdict changes when only the *layout* of the
request changes, with the identical text (CALM vs native format, original prompts). Such a change should not matter at
all, so a perturbation that changes fewer verdicts than it cannot be called a meaningful bias.

Classification from the 95% interval of the excess: lower bound above δ = **B** (bias), upper bound below δ = **-**
(no meaningful bias), otherwise **?** (inconclusive).

In [ ]:
rows = explore.design.build_design(citations=explore.authority.load_citations())
orig = {r["id"]: r for r in rows if r["bias"] == "original"}
layout = {}
for model in [x for x in run.MODELS if "mercury" not in x]:
    v = {f: {x["id"]: metrics._picked(orig[x["id"]], x) for x in run.load_records(model, f)
             if x["repeat"] == 0 and x["id"] in orig} for f in ["calm", "native"]}
    ids = [i for i in orig if v["calm"].get(i) and v["native"].get(i)]
    layout[model.split("/")[1].replace("-experimental", "")] = np.mean([v["calm"][i] != v["native"][i] for i in ids])
layout = pd.Series(layout, name="verdicts changed by layout alone")
display(layout.round(3)); print("median %.3f" % layout.median())

def classify(r, delta):
    return "B" if r.excess_lo > delta else "-" if r.excess_hi < delta else "?"
c = res.copy()
for delta in [0.02, 0.05, 0.10]:
    c[f"delta {delta}"] = c.apply(classify, axis=1, delta=delta)
c["own layout"] = [classify(r, layout[r.model]) for r in c.itertuples()]
print("CALM format, delta = 0.05"); display(c.pivot(index="bias", columns="model", values="delta 0.05"))
print("number of models with a bias (B)")
pd.DataFrame({k: c.groupby("bias")[k].apply(lambda s: (s == "B").sum()) for k in ["delta 0.02", "delta 0.05", "delta 0.1", "own layout"]})

jev-1.13                 0.008
kev-4b                   0.069
tev1-4b                  0.047
solar-decide             0.104
d1                       0.040
pplx-decider-v1.1-27b    0.017
clef                     0.044
clef-flash               0.094
Name: verdicts changed by layout alone, dtype: float64

median 0.045
CALM format, delta = 0.05


model,clef,clef-flash,d1,jev-1.13,kev-4b,pplx-decider-v1.1-27b,solar-decide,tev1-4b
bias,,,,,,,,
authority,B,B,B,B,B,B,?,B
"authority, better response",B,?,B,B,B,B,B,B
bandwagon,B,B,?,B,B,B,?,?
compassion,?,?,B,-,B,?,?,B
distraction,-,?,-,?,?,-,-,?
diversity,-,?,?,-,B,-,-,?
"diversity, neutral note",-,?,?,-,?,-,?,?
fallacy,-,B,?,-,-,?,?,?
position,B,B,B,B,B,B,B,B


number of models with a bias (B)


,delta 0.02,delta 0.05,delta 0.1,own layout
bias,,,,
authority,8,7,4,6
"authority, better response",8,7,6,6
bandwagon,7,5,4,5
compassion,7,3,2,4
distraction,4,0,0,1
diversity,4,1,0,2
"diversity, neutral note",4,0,0,1
fallacy,5,1,0,2
position,8,8,5,8


## 10. Which of the six criteria react? (decomposed format)

In the decomposed format each criterion is its own question. Mean shift of the targeted response's probability per
criterion and test, all models. If a cue acted on one criterion only (say, tone on helpfulness), it would show here.

In [ ]:
crit = explore.criterion_shifts()
crit = crit[crit.bias != "fallacy"]
crit.pivot_table(index="bias", columns="criterion", values="shift", aggfunc="mean").round(3)

criterion,accuracy,creativity,depth,detail,helpfulness,relevance
bias,,,,,,
authority,-0.102,-0.088,-0.075,-0.070,-0.096,-0.100
"authority, better response",-0.149,-0.127,-0.116,-0.109,-0.143,-0.140
bandwagon,0.086,0.093,0.084,0.084,0.099,0.092
compassion,0.042,0.041,0.042,0.040,0.045,0.041
distraction,-0.022,-0.024,-0.020,-0.020,-0.024,-0.023
diversity,-0.029,-0.028,-0.024,-0.022,-0.031,-0.030
"diversity, neutral note",-0.025,-0.023,-0.018,-0.018,-0.026,-0.025
position,0.038,0.045,0.039,0.033,0.038,0.040
sentiment,-0.247,-0.145,-0.228,-0.218,-0.258,-0.259


## 11. Ensembles of typed models

Can combining models reduce bias, since some models may be biased where others are not? From the stored answers
(CALM format, requests that all 8 models answered), with `judgebias/ensembles.py`:
* **Combinations of all 8:** mean, median and mean log-odds of P(A), and the majority of the verdicts. Each is read like
  a single model (verdicts, 50/50 rule, RR, CR, excess). Cost = sum of the models' costs, latency = slowest model.
* **All 255 subsets:** to avoid choosing on the test data, the subset with the smallest mean excess is chosen on one
  half of the items and evaluated on the other half.
* **One model, both orders:** on the items of the position test, the probabilities of the original and the swapped
  order are averaged, which removes position bias by construction.

In [ ]:
from judgebias import ensembles as E
M = E.Matrix("calm")
J = E.judges(M, pd.read_csv(CODE / "tables" / "time_and_cost.csv").query("format == 'calm'").rename(columns={"model": "judge"}))
J.to_csv(CODE / "tables" / "ensembles.csv", index=False)
J[["judge", "mean excess", "fallacy", "accuracy", "position", "compassion", "distraction", "sentiment", "bandwagon",
   "authority", "authority, better response", "usd_per_1000", "latency_s"]].round(3)

,judge,mean excess,fallacy,accuracy,position,compassion,distraction,sentiment,bandwagon,authority,"authority, better response",usd_per_1000,latency_s
0,jev-1.13,0.109,0.020,0.777,0.085,0.021,0.054,0.366,0.200,0.123,0.291,0.045,0.252
1,kev-4b,0.134,0.010,0.647,0.191,0.150,0.065,0.389,0.191,0.151,0.224,0.033,1.901
2,tev1-4b,0.117,0.071,0.656,0.243,0.210,0.047,0.254,0.045,0.147,0.233,0.036,0.559
3,solar-decide,0.078,0.058,0.667,0.155,0.078,0.023,0.241,0.032,0.085,0.120,0.055,2.866
4,d1,0.117,0.036,0.752,0.123,0.080,0.030,0.346,0.067,0.181,0.297,0.031,0.378
5,pplx-decider-v1.1-27b,0.112,0.044,0.749,0.077,0.043,0.019,0.293,0.260,0.163,0.380,0.017,0.376
6,clef,0.118,0.032,0.752,0.139,0.052,0.010,0.290,0.273,0.167,0.342,0.205,0.532
7,clef-flash,0.106,0.118,0.649,0.194,0.071,0.042,0.272,0.119,0.118,0.078,0.077,0.471
8,"all 8, mean",0.096,0.032,0.722,0.082,0.046,0.026,0.323,0.159,0.125,0.276,0.498,2.866
9,"all 8, median",0.105,0.028,0.740,0.071,0.071,0.031,0.333,0.186,0.111,0.313,0.498,2.866


In [ ]:
S = E.subsets(M)
S.to_csv(CODE / "tables" / "ensemble_subsets.csv", index=False)
best_single = S[S.k == 1].sort_values("excess_A").iloc[0]
best_subset = S.sort_values("excess_A").iloc[0]
print(f"chosen on half A, tested on half B: best single {best_single.subset} {best_single.excess_B:.3f}, "
      f"best subset {best_subset.subset} {best_subset.excess_B:.3f}")
display(S.groupby("k").excess_B.agg(["median", "min"]).round(3).T)
S.sort_values("excess_A").head(8).round(3)

chosen on half A, tested on half B: best single solar-decide 0.066, best subset jev-1.13+solar-decide 0.066


k,1,2,3,4,5,6,7,8
median,0.118,0.112,0.101,0.102,0.099,0.101,0.099,0.097
min,0.066,0.060,0.060,0.062,0.075,0.081,0.091,0.097


,subset,k,excess_A,excess_B,accuracy_A,accuracy_B
10,jev-1.13+solar-decide,2,0.061,0.066,0.679,0.688
37,jev-1.13+kev-4b+solar-decide,3,0.067,0.083,0.688,0.697
28,solar-decide+clef,2,0.071,0.075,0.725,0.719
100,jev-1.13+kev-4b+solar-decide+clef-flash,4,0.072,0.073,0.688,0.724
26,solar-decide+d1,2,0.074,0.060,0.665,0.697
119,jev-1.13+solar-decide+d1+clef-flash,4,0.074,0.081,0.706,0.738
65,kev-4b+solar-decide+clef-flash,3,0.076,0.083,0.661,0.679
29,solar-decide+clef-flash,2,0.076,0.068,0.665,0.656


In [ ]:
B = E.both_orders()
B.to_csv(CODE / "tables" / "both_orders.csv", index=False)
B.round(3)

,model,items,accuracy_one_order,accuracy_both_orders,change,change_lo,change_hi,orders_agree
0,jev-1.13,439,0.777,0.789,0.012,-0.006,0.032,0.897
1,kev-4b,439,0.647,0.645,-0.002,-0.030,0.025,0.809
2,tev1-4b,439,0.656,0.695,0.038,0.006,0.072,0.755
3,solar-decide,439,0.667,0.690,0.024,-0.006,0.054,0.753
4,d1,439,0.752,0.773,0.022,-0.003,0.047,0.873
5,pplx-decider-v1.1-27b,439,0.749,0.752,0.002,-0.016,0.021,0.923
6,clef,439,0.752,0.770,0.018,-0.007,0.046,0.861
7,clef-flash,439,0.649,0.672,0.023,-0.005,0.050,0.806


## 12. Do typed models and LLM judges find the same cues hard?

CALM released no verdicts per item, so we cannot compare the two kinds of judge item by item. We can compare their
*profiles*: per version of CALM's tests (its Table 8, a test without versions counts once, fallacy left out), the
median robustness rate and excess of the typed models and of CALM's six LLM judges, and the Spearman correlation
between the two. Versions of one test are not independent (eight of the 26 are sentiment), so the same correlation is
also given per test, with the versions of a test averaged. Descriptive, as the comparison with CALM is.

*Superseded (2026-10-09):* this section compares with CALM's published LLM numbers. The paper now uses our own LLM judges on the same prompts, `09_llm_analysis` sections 7 and 2.

In [ ]:
prof, cor = explore.profile_correlation()
prof.to_csv(CODE / "tables" / "profile_versions.csv", index=False)
cor.to_csv(CODE / "tables" / "profile_correlation.csv", index=False)
display(cor.round(3))
prof.round(3)

,level,measure,n,rho,p
0,version,RR,27,0.819,0.000
1,version,excess,27,0.741,0.000
2,test,RR,9,0.850,0.004
3,test,excess,9,0.667,0.050


,bias,variant,value_llm,value_typed,excess_llm,excess_typed
0,authority,Book,0.792,0.857,0.119,0.130
1,authority,Quote,0.752,0.807,0.143,0.193
2,authority,URL,0.834,0.872,0.065,0.108
3,bandwagon,60%,0.692,0.872,0.207,0.101
4,bandwagon,70%,0.672,0.863,0.217,0.137
5,bandwagon,80%,0.681,0.833,0.202,0.167
6,bandwagon,90%,0.708,0.800,0.196,0.191
7,compassion,-,0.865,0.924,0.042,0.074
8,distraction,h.c,0.749,0.962,0.147,0.033
9,distraction,h.r,0.821,0.951,0.089,0.035


## 13. Are the typed models closer in architecture also closer to the LLM judges?

Per typed model, the Spearman correlation between its profile over the 26 versions of CALM's tests (robustness rate
and excess, fallacy left out) and the median of CALM's six LLM judges. Architectures (Section 3.2): Tev1 keeps the
language-model head of its Qwen LLM and answers with one letter, Kev and the two Clef models add a head to a Qwen LLM,
Perplexity Decider is built on a Qwen LLM, Solar Decide runs on Solar Mini 4, Jev and D1 are undisclosed.
Exploratory: eight models, two of them of unknown architecture.

*Superseded (2026-10-09):* this section compares with CALM's published LLM numbers. The paper now uses our own LLM judges on the same prompts, `09_llm_analysis` sections 7 and 2.

In [ ]:
sim = explore.model_similarity()
sim.to_csv(CODE / "tables" / "model_similarity.csv", index=False)
sim.round(3)

,model,n,rho_RR,rho_excess
3,jev-1.13,27,0.918,0.854
4,kev-4b,27,0.839,0.792
5,pplx-decider-v1.1-27b,27,0.816,0.761
0,clef,27,0.818,0.751
2,d1,27,0.712,0.648
1,clef-flash,27,0.479,0.494
7,tev1-4b,27,0.504,0.456
6,solar-decide,27,0.737,0.371


## Notes

**First run, 2026-10-05.** Exploratory: patterns, not pre-planned tests. CALM format unless noted; fallacy left out.

* **Flips happen where the judge was unsure.** Flip rate 26 to 45% when the original was near 50/50 (margin < 0.1),
  3 to 10% when it was confident (margin 0.4 to 0.49), for every model. Mixed model: each +0.1 of margin halves the
  odds of a flip (odds ratio 0.46, 95% CI 0.45 to 0.47), with about the same slope for all 8 models (0.39 to 0.52).
  **Exception: sentiment**, which flips 26 to 30% even of confident verdicts; an emotional tone overrides certainty.
  The citation on the better response also flips 13 to 14% of confident verdicts.
* **Identity notes act like any note.** Against the neutral note, identity notes do not raise the odds of a flip
  (odds ratio 1.10, CI 0.98 to 1.24), and neither do distraction (1.02) or padding (1.01). The ranking of the others
  (odds ratios vs the neutral note): sentiment 18.7, citation on the better response 6.9, bandwagon 5.3, position
  4.7, citation on the worse response 3.1, compassion fade 2.5.
* **Models** (odds ratios vs Jev, same items and tests): Solar 3.3 (the noisy one), Clef 2.3, Clef Flash and
  Perplexity about 0.9 to 0.95, D1 0.60, Kev 0.54, Tev1 0.52.
* **Formats:** the decomposed format lowers the odds of a flip by 22% (OR 0.78, CI 0.75 to 0.81) against the CALM
  format; native is about the same (1.05).
* **Items:** item-level variation is substantial (ICC 0.16). Model-specific item effects (item x model, variance 1.30)
  are larger than effects shared by all models (item, 0.97), so each model has partly its own weak items.
  Correlation of per-item shifts between models: highest for sentiment (0.57) and authority (0.34 to 0.41), lowest for
  distraction (0.13) and compassion (0.16). For sentiment the *same* items move all models, which points to the items
  or their rewrites.
* **Question source:** among alignment items, Py-DPO (tested code against open-model code) flips least (bandwagon
  0.11 vs 0.20 to 0.26 for Orca-DPO and Emerton-DPO; mixed model OR 0.42 vs Emerton-DPO). Emerton-DPO pairs two
  GPT-4 responses, so the quality gap is small. Fits CALM's observation that bias is stronger when the two responses
  are close in quality. Fact sources do not differ.
* **Direction, seen in the flips:** citations, padding and tones push away from the response that carries them. A
  citation on the worse response flips 44% of verdicts that were on the worse response and 2% of those on the better
  one; on the better response it is the reverse (39% vs 2%). Padding the worse response flips 42% of the verdicts that
  were on it and 2% of the others. Bandwagon flips only verdicts that were on the better response (25% vs 2%).
* **Length** of the responses barely matters (odds ratio 0.97 per unit of log length ratio, CI 0.90 to 1.05).
* **Bandwagon dose:** the pull towards the worse response grows with the claimed majority for all 8 models
  (60% -> 90%: e.g. Jev +0.14 -> +0.20, Kev +0.05 -> +0.13, Clef +0.19 -> +0.30; slope 0.007 to 0.039 per 10
  points). CALM's LLMs showed no clear trend in robustness rate (their Figure 9c); the typed models read the number.
* **Variance of the shift** (crossed random effects, all formats): test 28%, test x item 8%, test x model 6%,
  item x model 2%, model 0.6%, format and source about 0, residual 55%. For the *size* of the shift: test 15%,
  item x model 9%, test x model 7%, test x item 6%, item 4%, model 0.7%. **Which perturbation matters far more than
  which typed model**; a model's overall level hardly matters, its reaction to specific tests does.
* **Model level** (8 models, descriptive): more consistent models show a larger excess (Spearman 0.68, driven by
  Solar's noise), accuracy is unrelated (0.29). A better judge is not a less biased one.
* **A minimum size for a bias, δ:** a pure layout change (CALM vs native, identical text) changes 0.8% (Jev) to 10.4%
  (Solar) of verdicts on the originals, median 4.5%. With δ = 0.05, from the interval of the excess: sentiment and
  position are biases in 8 of 8 models, both authority tests in 7, bandwagon 5, compassion 4, identity notes 1,
  verbosity 1, distraction 0. The same three groups appear at δ = 0.02, at δ = 0.10, and with each model's own layout
  sensitivity as δ: (1) tone and position everywhere, (2) citations, majority and model names in most models,
  (3) padding, distraction and identity notes hardly ever.
* **Ensembles (section 11, 2026-10-05).** Majority vote of all 8: mean excess 0.105 (median single model 0.127,
  best single Solar 0.084); mean, median, log-odds of P(A): 0.111 to 0.121. Position 0.050, below every single model
  (0.085 to 0.243), because six models prefer the second place and two the first; model names 0.026 vs median 0.079.
  Tone (0.33), citation on the better response (0.30), majority (0.16): at the level of the median model, since all
  models are pushed the same way. Accuracy of the vote 0.745 (members 0.647 to 0.777). Subsets chosen on half A and
  tested on half B: best subset (Jev + Solar) 0.075 vs best single (Solar) 0.071, larger subsets converge to 0.12.
  Cost of all 8: $0.52 per 1,000 (sum), latency 1.9 s (slowest model, Kev). One model in both orders (position test
  items): removes position bias by construction, accuracy up for 6 of 8 (Tev1 +4.5 points), down for Kev (-0.2) and
  Perplexity (-2.3). In the paper as Section 4.9.
* **Same cues for both kinds of judge (section 12, 2026-10-07).** Across the 26 versions of CALM's tests (fallacy left
  out), the cues that sway CALM's LLM judges most also sway the typed models most: Spearman 0.83 between the median
  robustness rates and 0.79 between the median excesses. Per test (8 tests, versions averaged): 0.81 and 0.76. Both
  kinds are moved most by a tone on the better response and least by a tone on the worse one. The typed models stand
  apart in padding, distraction and identity notes, which move them much less. In the paper in Section 4.4.
* **Architecture and similarity to LLM judges (section 13, 2026-10-07).** Spearman of each model's excess profile over
  the 26 versions with the median LLM judge: Jev 0.87, Perplexity 0.83, Clef 0.80, Kev 0.77, D1 0.64, Clef Flash 0.55,
  Tev1 0.51, Solar 0.42. Tev1, which keeps the language-model head of its LLM, is among the least LLM-like, so a closer
  architecture does not make a model behave more like an LLM judge. Exploratory, eight models. In the paper's appendix
  on differences between models.